In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    # This sets the key for any Python code that runs later
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ GOOGLE_API_KEY loaded successfully.")
except Exception as e:
    print("🔑 ERROR: Please add your 'GOOGLE_API_KEY' to Kaggle Secrets first.")
    print("   (Click Add-ons > Secrets > New Secret)")
    GOOGLE_API_KEY = None

✅ GOOGLE_API_KEY loaded successfully.


In [2]:
!pip install -q google-adk "uvicorn[standard]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.6/456.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.1/456.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 14.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<

In [3]:
# Create the agent project folder
!adk create math_agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY


Agent created in /kaggle/working/math_agent:
- .env
- __init__.py
- agent.py



In [4]:
%%writefile /kaggle/working/math_agent/agent.py

import os
from kaggle_secrets import UserSecretsClient
from google.genai import types

# --- Imports from your notebooks ---
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.code_executors import BuiltInCodeExecutor # Day 2a Concept
from google.adk.a2a.utils.agent_to_a2a import to_a2a      # Day 5a Concept

# --- 1. API Key and Model Setup ---
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
except Exception as e:
    pass # Key is already in environment

retry_config = types.HttpRetryOptions(attempts=5)
model = Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config)

# --- 2. Define the Agent (Day 2a) ---
# This agent's only job is to run math code.
math_agent = LlmAgent(
    model=model,
    name="MathAgent",
    instruction="""You are a calculator. You MUST NOT answer math questions.
    You MUST generate Python code to find the answer.
    Your code MUST print the final result.""",
    code_executor=BuiltInCodeExecutor(), 
)

# --- 3. Expose the Agent (Day 5a) ---
# This is the key step: we wrap the agent in an A2A server
# The `adk run` command will look for this `app` variable.
app = to_a2a(
    math_agent, 
    port=8001  # We will run this server on port 8001
)

print("✅ 'math_agent/agent.py' file written successfully.")

Overwriting /kaggle/working/math_agent/agent.py


In [5]:
!adk create coordinator_agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY


Agent created in /kaggle/working/coordinator_agent:
- .env
- __init__.py
- agent.py



In [6]:
%%writefile /kaggle/working/coordinator_agent/agent.py

import os
from kaggle_secrets import UserSecretsClient
from google.genai import types

# --- Imports from ALL your notebooks ---
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool, google_search
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
    AGENT_CARD_WELL_KNOWN_PATH,
)

# --- 1. API Key and Model Setup ---
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
except Exception as e:
    pass

retry_config = types.HttpRetryOptions(attempts=5)
model = Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config)

# --- 2. Define Custom Tools (Day 2a) ---
def exit_loop():
    """Call this function ONLY when the report is 'APPROVED'."""
    print("--- 🔁 Loop: 'APPROVED' received. Exiting loop. ---")
    return {"status": "approved", "message": "Report approved."}

# --- 3. Define ALL Specialist Agents (Day 1b) ---

# --- Step 1a: Parallel Researcher (Tech) ---
tech_researcher = LlmAgent(
    model=model,
    name="TechResearcher",
    instruction="""You are a tech analyst. 
    Research the user's topic from a technical perspective using google_search.
    Find 2-3 key technical challenges or innovations.
    Output a bulleted list.""",
    tools=[google_search],
    output_key="tech_research",
)

# --- Step 1b: Parallel Researcher (Finance) ---
finance_researcher = LlmAgent(
    model=model,
    name="FinanceResearcher",
    instruction="""You are a financial analyst.
    Research the user's topic from a financial perspective using google_search.
    Find 2-3 key costs, investments, or market impacts.
    Output a bulleted list.""",
    tools=[google_search],
    output_key="finance_research",
)

# --- Step 2a: Loop Agent (Writer) ---
# --- THIS IS THE FIX ---
# The {critique} variable has been REMOVED from this agent's instructions.
# This agent's job is ONLY to write the *first* draft.
writer_agent = LlmAgent(
    model=model,
    name="WriterAgent",
    instruction="""You are a report writer.
    Tech Research: {tech_research}
    Finance Research: {finance_research}

    Write a short (150-word) report combining all information.
    This is the first draft, so just present the information clearly.""",
    output_key="draft",
)

# --- Step 2b: Loop Agent (Critic) ---
# This agent runs *inside* the loop and CREATES the {critique} variable.
critic_agent = LlmAgent(
    model=model,
    name="CriticAgent",
    instruction="""You are a critic. Review the draft: {draft}
    - If it's perfect, respond with the exact word: "APPROVED"
    - If it's missing hard numbers (like costs, or calculations),
      respond with: "NEEDS_CALCULATION"
    - Otherwise, provide 1-2 bullet points of feedback.""",
    output_key="critique",
)

# --- Step 2c: Loop Agent (Refiner) & A2A Tool (Day 5a) ---
# This agent runs *inside* the loop and USES the {critique} variable.
remote_math_agent = RemoteA2aAgent(
    name="math_agent",
    description="A specialist agent that performs math calculations.",
    agent_card=f"http://127.0.0.1:8001{AGENT_CARD_WELL_KNOWN_PATH}",
)

refiner_agent = LlmAgent(
    model=model,
    name="RefinerAgent",
    instruction="""You are a refiner. You have a draft and a critique.
    Draft: {draft}
    Critique: {critique}

    Your task is to analyze the critique:
    - IF critique is "APPROVED", call `exit_loop`.
    - IF critique is "NEEDS_CALCULATION", use the `math_agent` sub-agent to
      calculate a key financial number (e.g., 5-year cost projection)
      and add it to the report.
    - OTHERWISE, rewrite the draft to fix the critique.
    """,
    output_key="draft",
    tools=[
        FunctionTool(exit_loop)
    ],
    sub_agents=[
        remote_math_agent      
    ]
)

# --- 4. Assemble the Final Workflow (Day 1b) ---
parallel_step = ParallelAgent(
    name="ResearchTeam",
    sub_agents=[tech_researcher, finance_researcher],
)
loop_step = LoopAgent(
    name="RefinementLoop",
    sub_agents=[
        critic_agent,
        refiner_agent,
    ],
    max_iterations=3,
)

# This is the 'root_agent' that adk web will find
root_agent = SequentialAgent(
    name="CoordinatorPipeline",
    sub_agents=[
        parallel_step,  # Step 1: Do Parallel Research
        writer_agent,   # Step 2: Write First Draft
        loop_step,      # Step 3: Run Refinement Loop
    ],
)

print("✅ 'coordinator_agent/agent.py' file written successfully. (FIXED)")

Overwriting /kaggle/working/coordinator_agent/agent.py


In [7]:
import subprocess
import time
import requests

print("🚀 Starting Server 1 (MathAgent on port 8001) using Uvicorn...")

server_process = subprocess.Popen(
    [
        "uvicorn",
        "agent:app",  # Looks for the 'app' variable in 'agent.py'
        "--host", "127.0.0.1",  # <-- FIX 1: Force IPv4
        "--port", "8001",
        "--app-dir", "/kaggle/working/math_agent/" # Tells uvicorn where to find agent.py
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ}, # Pass the GOOGLE_API_KEY
)

print("...Waiting 30 seconds for server to boot...") # <-- FIX 2: Wait longer
time.sleep(30)

# --- Verify Server 1 is running ---
try:
    response = requests.get("http://127.0.0.1:8001/.well-known/agent-card.json")
    if response.status_code == 200:
        print("✅ SUCCESS: Server 1 (MathAgent) is running and A2A card is accessible.")
    else:
        print(f"❌ FAILED: Server 1 returned status {response.status_code}.")
        print("--- Server Error Log ---")
        print(server_process.stderr.read().decode())
except Exception as e:
    print(f"❌ FAILED: Could not connect to Server 1. {e}")
    print("--- Server Error Log ---")
    stderr_output = server_process.stderr.read().decode()
    if stderr_output:
        print(stderr_output)
    else:
        print("No error output from server. The process may have failed to start.")
        stdout_output = server_process.stdout.read().decode()
        if stdout_output:
            print("--- Server Standard Log ---")
            print(stdout_output)

🚀 Starting Server 1 (MathAgent on port 8001) using Uvicorn...
...Waiting 30 seconds for server to boot...
✅ SUCCESS: Server 1 (MathAgent) is running and A2A card is accessible.


In [8]:
from IPython.core.display import display, HTML
try:
    from jupyter_server.serverapp import list_running_servers
    servers = list(list_running_servers())
    if servers:
        base_url = servers[0]['base_url']
        url_prefix = base_url + 'proxy/8000/'
        display(HTML(f'<a href="{url_prefix}" target="_blank">Open ADK Web UI</a>'))
    else:
        url_prefix = 'http://localhost:8000/'
        print('Running in standalone mode. Access ADK UI at http://localhost:8000')
except Exception as e:
    url_prefix = 'http://localhost:8000/'
    print(f'ADK local endpoint configured at {url_prefix} (fallback mode: {e})')


ADK local endpoint configured at http://localhost:8000/ (standalone mode)


In [ ]:
!adk web --url_prefix {url_prefix} \
         --session_service_uri "sqlite:////kaggle/working/coordinator_chat.db" \
         --log_level DEBUG